# Exploration code 1
## Universidad ICESI 
### David Mauricio Orozco Rios
### author: Davoroz06 - IG

In [ ]:
# libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import gmean
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# options for data display
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text


In [ ]:
# Filter function
def filter_products(df, min_observations_per_month, min_months):
    # Group by 'descripcion' and get the count of observations per month
    df['month'] = df['fecha'].dt.to_period('M')  # Create a 'month' column
    monthly_counts = df.groupby(['descripcion', 'month']).size().reset_index(name='count')
    monthly_counts['count_gt_min'] = monthly_counts['count'] > min_observations_per_month

    # Count number of months that pass the minimum criteria
    min_monthsdf = monthly_counts.groupby(['descripcion']).agg(months_true=('count_gt_min', 'sum')).reset_index()

    filtered_items = min_monthsdf[min_monthsdf['months_true']>min_months]['descripcion']

    # Get the descriptions that meet the criteria
    valid_descriptions = filtered_items.unique()

    # Filter the original DataFrame to keep only valid descriptions
    filtered_df = df[df['descripcion'].isin(valid_descriptions)]

    # Drop extra columns
    filtered_df = filtered_df.drop("month", axis = 1)

    return filtered_df

# Transition matrix function
def calculate_transition_matrix(df):
    # Define states based on whether price equals the regular price
    df['State'] = df.apply(lambda row: 1 if pd.notnull(row['precio']) and pd.notnull(row['Regular_Price']) and row['precio'] == row['Regular_Price'] else (2 if pd.notnull(row['precio']) and pd.notnull(row['Regular_Price']) else None), axis=1)
    #df['State'] = df.apply(lambda row: 1 if row['precio'] == row['Regular_Price'] else 2, axis=1)
    
    # Sort by tienda, descripcion and fecha to ensure correct time ordering
    df = df.sort_values(by=['tienda', 'descripcion', 'fecha'])
    
    # Create columns for previous state (shifted by one row)
    df['Prev_State'] = df.groupby(['tienda', 'descripcion'])['State'].shift(1)
    
    # Create a transition table that counts the occurrences of each transition
    transition_counts = pd.crosstab(df['Prev_State'], df['State'])
    
    # Calculate the transition matrix as probabilities by dividing by row sums
    transition_matrix = transition_counts.div(transition_counts.sum(axis=1), axis=0)
    
    # To a DataFrame
    transition_matrix = pd.DataFrame(transition_matrix)

    # Get rows and columns
    rows, columns = transition_matrix.shape

    if rows < 2:
        # Adding a new row and a new column with zero values
        transition_matrix.loc['Y'] = 0      # Add a new row with all zeroes
    if columns < 2:
        transition_matrix['B'] = 0          # Add a new column with all zeroes

    # Add labels to the rows and columns of the transition matrix
    transition_matrix.index = ['Regular_Price', 'Non_Regular_Price']
    transition_matrix.columns = ['Regular_Price', 'Non_Regular_Price']
    
    # Calculate week transition matrix (T^7)
    week_transition_matrix = np.array(np.linalg.matrix_power(transition_matrix, 7))
    
    # Add labels
    week_transition_matrix = pd.DataFrame(week_transition_matrix, index=["Regular_price", "Non_Regular_Price"], columns=["Regular_price", "Non_Regular_Price"])
    
    # Calculate month transition matrix (T^30)
    month_transition_matrix = np.array(np.linalg.matrix_power(transition_matrix, 30))
    
    # Add labels
    month_transition_matrix = pd.DataFrame(month_transition_matrix, index=["Regular_price", "Non_Regular_Price"], columns=["Regular_price", "Non_Regular_Price"])
    
    # Return the matrices
    return transition_matrix, week_transition_matrix, month_transition_matrix

# Function to calculate relevant statistics
def calculate_price_statistics_for_product(df):
    # Ensure fecha is in datetime format
    df['fecha'] = pd.to_datetime(df['fecha'])
    
    # Create a 'YearMonth' column for monthly grouping
    df['YearMonth'] = df['fecha'].dt.to_period('M')
    
    # 1. Fraction of days spent at reference prices
    fraction_at_reference = (df['Price_type'] == 'Regular price').mean()
    
    # 2. Fraction of days spent below the reference price (Sales Price)
    fraction_nonreference_below = ((df['Price_type'] == 'Sales price').sum())/((df['Price_type'] != 'Regular price').sum())
    
    # 3. Fraction of months in which daily prices are equal for the whole month
    def is_price_constant(group):
        return group['precio'].nunique() == 1  # True if all prices are the same

    price_constant_by_month = df.groupby('YearMonth').apply(is_price_constant)
    fraction_constant_months = price_constant_by_month.mean()
    
    # 4. Fraction of price changes that are from a non-reference price to a reference price
    df = df.sort_values(by=['fecha'])
    
    # Define state: 1 if 'Regular price', 0 if otherwise
    df['Is_Reference'] = (df['Price_type'] == 'Regular price').astype(int)
    
    # Calculate transitions within the product
    df['Prev_Is_Reference'] = df['Is_Reference'].shift(1)
    
    # Count transitions from non-reference to reference price (from 0 to 1)
    non_ref_to_ref_changes = ((df['Prev_Is_Reference'] == 0) & (df['Is_Reference'] == 1) & (df['precio'] != df['precio'].shift(1))).sum()
    total_price_changes = ((df['precio'] != df['precio'].shift(1)).sum() - 1)
    
    fraction_non_ref_to_ref = non_ref_to_ref_changes / total_price_changes if total_price_changes > 0 else 0
    
    # Return statistics as a Series
    return pd.Series({
        'Fraction of days at reference prices': round(fraction_at_reference, 2),
        'Fraction of non reference price below reference prices (Sales)': round(fraction_nonreference_below, 2),
        'Fraction of months with constant daily prices': round(fraction_constant_months, 2),
        'Fraction of price changes from non-reference to reference': round(fraction_non_ref_to_ref, 2)
    })


In [ ]:
# load retailer data
data = pd.read_csv(wd_db + "Retailer_data.csv")

# Convert 'fecha' to datetime format
data['fecha'] = pd.to_datetime(data['fecha'])

In [ ]:
# Drop previous index column
data = data.drop(["Unnamed: 0"], axis = 1)

In [ ]:
# Drop zero prices
data = data[data["precio"] > 0]

1. Errores
2. Comparar con otra literatura
3. Pensar como ajustar las matrices pensando en la frecuencia de los datos

In [ ]:
# Usage of the function
min_observations_per_month, min_months = 4, 2 

data_1 = filter_products(data, min_observations_per_month, min_months)
display(data_1)

In [ ]:
# Define your minimum and maximum dates
min_date = min(data_1[data_1["tienda"]=="B"]["fecha"])
max_date = max(data_1[data_1["tienda"]=="B"]["fecha"])

# Filter the dataframe
data_1 = data_1[(data_1['fecha'] >= min_date) & (data_1['fecha'] <= max_date)]

# Define the full date range
date_range = pd.date_range(start=data_1['fecha'].min(), end=data_1['fecha'].max())

# Get unique combinations of 'descripcion' and 'tienda'
product_store_pairs = data_1[['descripcion', 'tienda']].drop_duplicates()

# Crear un DataFrame de la lista y agregarlo a cada fila del DataFrame original
datesdf = pd.DataFrame({'fecha': date_range}).merge(product_store_pairs, how='cross')

# Add missing values to DataFrame
data_1 = datesdf.merge(data_1, on = ['tienda', 'descripcion', 'fecha'], how = 'left')

### Estadisticas descriptivas

In [ ]:
# Convertir la columna de fecha a formato de fecha y agregar una columna de mes
data_1['fecha'] = pd.to_datetime(data_1['fecha'])
data_1['mes'] = data_1['fecha'].dt.to_period('M')

# Calcular las estadísticas descriptivas por tienda
stats = data_1[~data_1["precio"].isna()].groupby('tienda').apply(lambda x: pd.Series({
    'Products': x['descripcion'].nunique(),
    'Observations_Month': x.groupby(['mes', 'descripcion']).size().mean(),
    'Mean_price': x['precio'].mean(),
    'Median_price': x['precio'].median(),
    'Mode_price': x['precio'].mode()[0] if not x['precio'].mode().empty else None,
    'Standard_deviation': x['precio'].std(),
    'Date_range': (x['fecha'].max() - x['fecha'].min()).days
}))

# Format numeric columns to integers (no decimals)
stats['Products'] = stats['Products'].astype(int)
stats['Observations_Month'] = stats['Observations_Month'].apply(lambda x: f"{x:,.2f}")
stats['Date_range'] = stats['Date_range'].astype(int)

# Format price columns to $ with 2 decimal places
stats['Mean_price'] = stats['Mean_price'].apply(lambda x: f"${x:,.2f}")
stats['Median_price'] = stats['Median_price'].apply(lambda x: f"${x:,.2f}")
stats['Mode_price'] = stats['Mode_price'].apply(lambda x: f"${x:,.2f}" if x is not None else None)
stats['Standard_deviation'] = stats['Standard_deviation'].apply(lambda x: f"${x:,.2f}")

pd.DataFrame(stats)



### 1. Eichenbum regular price

In [ ]:
# Group by, then calculate mode for each group
def mode_price(group):
    return group.mode()[0] if not group.mode().empty else None

# Regular price function
def calculate_regular_price_eichenbaum(df):
    # Ensure fecha is in datetime format
    df['fecha'] = pd.to_datetime(df['fecha'])
    
    # Create a 'YearMonth' column for grouping
    df['YearMonth'] = df['fecha'].dt.to_period('M')

    # Apply the mode calculation for each product and month
    regular_prices = df.groupby(['tienda', 'descripcion', 'YearMonth'])['precio'].apply(mode_price).reset_index()
    
    # Merge regular price back into the original dataframe
    df = df.merge(regular_prices, on=['tienda', 'descripcion', 'YearMonth'], how='left', suffixes=('', '_regular'))
    
    # Rename the new column for clarity
    df.rename(columns={'precio_regular': 'Regular_Price'}, inplace=True)
    
    # Drop the 'YearMonth' column as it's no longer needed
    df.drop(columns='YearMonth', inplace=True)
    
    return df

In [ ]:
eidf = calculate_regular_price_eichenbaum(data_1)

In [ ]:
eidf[eidf["descripcion"] == eidf["descripcion"][0]]

In [ ]:
# Define the 'Price type' column based on comparison with Regular_Price
eidf['Price_type'] = eidf.apply(lambda row: 'Regular price' if row['precio'] == row['Regular_Price'] 
                            else 'Sales price' if row['precio'] < row['Regular_Price']
                            else 'Higher price', axis=1)

#### Transition matrix 

Define a filter to erase missing products

In [ ]:
stores = ["A1", "A2", "C", "B"]
for store in stores:
    print("Results in store " + store + ": \t")
    transition_matrix, two_step_matrix, three_step_matrix = calculate_transition_matrix(eidf[eidf["tienda"]== store])
    # Print results
    print("Daily Transition Matrix:")
    print(transition_matrix)
    print("----------")

    print("Weekly Transition Matrix:")
    print(two_step_matrix)
    print("----------")

    print("Monyhly Transition Matrix:")
    print(three_step_matrix)
    print("----------")

Statistic table per product

In [ ]:
%%capture
by_product_stats = eidf.groupby(['tienda', 'descripcion']).apply(calculate_price_statistics_for_product)

In [ ]:
# Get the unique values from the two columns
unique_values = eidf[['tienda', 'descripcion']].drop_duplicates()

# Filter the original DataFrame using the unique values
by_store_stats = by_product_stats.merge(unique_values, on=['tienda', 'descripcion'])

In [ ]:
# granular, types of retailers, explain more about purchase  
# Summary results by store
summary_store_results = by_store_stats.groupby('tienda')[['Fraction of days at reference prices',
                                                          'Fraction of non reference price below reference prices (Sales)',
                                                          'Fraction of months with constant daily prices',
                                                          'Fraction of price changes from non-reference to reference']].agg(
    ["mean", "median"]  # Apply any other functions you need
).reset_index()

# Display the result
display(summary_store_results)

In [ ]:
# List of products to plot
selected_products = eidf[eidf["Price_type"]=="Sales price"][["tienda", "descripcion", "Price_type"]].value_counts()

# Create the plot
plt.figure(figsize=(10, 6))
fig, ax = plt.subplots(figsize=(10, 6))  # Create fig and ax to work with ax

for (tienda, descripcion, price), count in selected_products[0:2].items():
    # Filter dataframe for selected products
    product_data = eidf[(eidf['descripcion'] == descripcion) & (eidf['tienda'] == tienda)]
    
    # Plot price and regular price
    ax.plot(product_data['fecha'], product_data['precio'], label=f'Price - {tienda + "-" + descripcion}', marker='o')
    ax.plot(product_data['fecha'], product_data['Regular_Price'], label=f'Regular Price - {tienda + "-" + descripcion}', linestyle='--')

# Customize the plot
ax.set_xlabel('Date')
ax.set_ylabel('Price')

# Format y-axis as currency
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

# Additional plot customizations
ax.set_title('Price and Regular Price for Selected Products')
# Place legend below the plot
ax.legend(bbox_to_anchor=(0.5, -0.2), loc='upper center', borderaxespad=0., ncol=1)
ax.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()

# Show the plot
plt.show()

### 2. Nakamura (in five facts about prices)

0. if $p_t = r_{t-1}$ then $r_t = r_{t-1}$.
1. if $p_t > r_{t-1}$ then $r_t = p_{t}$.
2. if $r_{t-1} \in \{p_{t+1},\dots,p_{t+j}\}$ and the price never rises above $r_{t-1}$ before returning to $r_{t-1}$, then $r_t = r_{t-1}$.
3. If the set $\{p_{t}, p_{t+1}, \dots, p_{t+L}\}$ has K or more different elements, then $r_t = p_t$.
4. Define $p_{max} = max\{p_{t}, p_{t+1}, \dots, p_{t+L}\}$ and $t_{max} = \text{first-time } max\{p_{t}, p_{t+1}, \dots, p_{t+L}\}$. If $p_{max} \in \{p_{tmax+1}, ..., p_{tmax+L}\}$, then $r_{t} = p_{max}$.
5. r_t = p_t.

In the first time period, the algorithm begins at step 3 (the first step that does not refer to a previous regular price).

In [ ]:
def regular_price_nakamura(col_p, L = 1, K = 1, J = 5):
    # Regular price list
    r_list = []
    for i in range(len(col_p)):
        if(pd.isna(col_p[i])):
            r_list.append(None)
            continue
        # If we are in the first time period
        if i == 0:
            # Third step
            if len(set(col_p[i:(i+(L+1))])) >= K:
                r = col_p[i]
                r_list.append(r)
                continue
            # Fourth step
            p_max = max(col_p[i:(i+(L+1))])
            t_max = next((j for j, x in enumerate(col_p) if x == p_max), None)
            list_max = col_p[(t_max+1):(t_max+L)]
            if p_max in list_max:
                r = p_max
                r_list.append(r)
                continue
            # fifth step
            r = col_p[i]
            r_list.append(r)
        # For second period onwards
        else:
            # Zero step
            if col_p[i] == r_list[(i-1)]:
                r = r_list[(i-1)]
                r_list.append(r)
                continue
            # First step
            if (r_list[(i-1)] is not None) and (col_p[i] is not None) and (col_p[i] > r_list[(i-1)]):
                r = col_p[i]
                r_list.append(r)
                continue
            # Second step
            list_second = col_p[(i+1):(i+1+J)]
            if (r_list[(i-1)] in list_second):
                reverse_second = next((j for j, x in enumerate(reversed(list_second)) if x == r_list[(i-1)]), None)
                if reverse_second is not None:    
                    t_second = len(list_second) - 1 - reverse_second
                else:
                    t_second = None
                if any(x <= r_list[(i-1)] for x in list_second):
                    r = r_list[(i-1)]
                    r_list.append(r)
                    continue
            # Third step
            if len(set(col_p[i:(i+(L+1))])) >= K:
                r = col_p[i]
                r_list.append(r)
                continue
            # Fourth step
            p_max = max(col_p[i:(i+(L+1))])
            t_max = next((j for j, x in enumerate(col_p) if x == p_max), None)
            list_max = col_p[(t_max+1):(t_max+L)]
            if p_max in list_max:
                r = p_max
                r_list.append(r)
                continue
            # fifth step
            r = col_p[i]
            r_list.append(r)
    return r_list

In [ ]:
# Regular price function
def calculate_regular_price_nakamura(df, l, k, j):
    # Ensure fecha is in datetime format
    df['fecha'] = pd.to_datetime(df['fecha'])
    
    # Sort by tienda, descripcion and fecha to ensure correct time ordering
    df = df.sort_values(by=['tienda', 'descripcion', 'fecha'])
    
    # Apply to a DataFrame
    df['Regular_Price'] = df.groupby(['tienda', 'descripcion'])['precio'].transform(lambda x: regular_price_nakamura(x.tolist(), L = l, K = k, J = j))
    
    return df

In [ ]:
# Data Frame for Example
selected_products = data_1[["tienda", "descripcion"]].value_counts()

for (tienda, descripcion), count in selected_products[1:2].items():
    # Filter dataframe for selected products
    exampledf = data_1[(data_1['descripcion'] == descripcion) &
                            (data_1['tienda'] == tienda)]

calculate_regular_price_nakamura(exampledf, l = 1, k = 1, j = 5)

In [ ]:
nadf = calculate_regular_price_nakamura(data_1, l = 1, k = 1, j = 5)

In [ ]:
# Define the 'Price type' column based on comparison with Regular_Price
nadf['Price_type'] = nadf.apply(lambda row: 'Regular price' if row['precio'] == row['Regular_Price'] 
                            else 'Sales price' if row['precio'] < row['Regular_Price']
                            else 'Higher price' if row['precio'] > row['Regular_Price']
                            else None, axis=1)

In [ ]:
nadf[nadf["precio"]!=nadf["Regular_Price"]].Price_type.value_counts()

#### Transition matrix 

Define a filter to erase missing products

In [ ]:
stores = ["A1", "A2", "C", "B"]
for store in stores:
    print("Results in store " + store + ": \t")
    transition_matrix, two_step_matrix, three_step_matrix = calculate_transition_matrix(nadf[nadf["tienda"]== store])
    # Print results
    print("One-Step Transition Matrix:")
    print(transition_matrix)
    print("----------")

    print("Weekly Transition Matrix:")
    print(two_step_matrix)
    print("----------")

    print("Monthly Transition Matrix:")
    print(three_step_matrix)
    print("----------")

Statistic table per product

In [ ]:
%%capture
by_product_stats = nadf.groupby(['tienda', 'descripcion']).apply(calculate_price_statistics_for_product)

In [ ]:
# Get the unique values from the two columns
unique_values = nadf[['tienda', 'descripcion']].drop_duplicates()

# Filter the original DataFrame using the unique values
by_store_stats = by_product_stats.merge(unique_values, on=['tienda', 'descripcion'])

In [ ]:
# granular, types of retailers, explain more about purchase  
# Summary results by store
summary_store_results = by_store_stats.groupby('tienda')[['Fraction of days at reference prices',
                                                          'Fraction of non reference price below reference prices (Sales)',
                                                          'Fraction of months with constant daily prices',
                                                          'Fraction of price changes from non-reference to reference']].agg(
    ["mean", "median"]  # Apply any other functions you need
).reset_index()

# Display the result
display(summary_store_results)

In [ ]:
# List of products to plot
selected_products = nadf[nadf["Price_type"]=="Sales price"][["tienda", "descripcion", "Price_type"]].value_counts()

# Create the plot
plt.figure(figsize=(10, 6))
fig, ax = plt.subplots(figsize=(10, 6))  # Create fig and ax to work with ax

for (tienda, descripcion, price), count in selected_products[0:2].items():
    # Filter dataframe for selected products
    product_data = nadf[(nadf['descripcion'] == descripcion) & (nadf['tienda'] == tienda)]
    
    # Plot price and regular price
    ax.plot(product_data['fecha'], product_data['precio'], label=f'Price - {tienda + "-" + descripcion}', marker='o')
    ax.plot(product_data['fecha'], product_data['Regular_Price'], label=f'Regular Price - {tienda + "-" + descripcion}', linestyle='--')

# Customize the plot
ax.set_xlabel('Date')
ax.set_ylabel('Price')

# Format y-axis as currency
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

# Additional plot customizations
ax.set_title('Price and Regular Price for Selected Products')
ax.legend()
ax.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# List of products to plot
selected_products = nadf[nadf["Price_type"]=="Sales price"][["tienda", "descripcion", "Price_type"]].value_counts()
selected_products

In [ ]:
# List of products to plot
selected_products = eidf[eidf["Price_type"] == "Sales price"][["tienda", "descripcion", "Price_type"]].value_counts()

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8))  # Create fig and ax to work with ax

for (tienda, descripcion, price), count in selected_products[0:2].items():
    # Filter dataframe for selected products
    product_eidf = eidf[(eidf['descripcion'] == descripcion) & (eidf['tienda'] == tienda)]
    product_nadf = nadf[(nadf['descripcion'] == descripcion) & (nadf['tienda'] == tienda)]
    
    # Plot price and regular price
    ax.plot(product_eidf['fecha'], product_eidf['precio'], label=f'Price - {tienda + "-" + descripcion}', marker='o')
    ax.plot(product_eidf['fecha'], product_eidf['Regular_Price'], label=f'Eichenbaum regular - {tienda + "-" + descripcion}', linestyle='--')
    ax.plot(product_nadf['fecha'], product_nadf['Regular_Price'], label=f'Nakamura regular - {tienda + "-" + descripcion}', linestyle='--')

# Customize the plot
ax.set_xlabel('Date')
ax.set_ylabel('Price')

# Format y-axis as currency
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

# Additional plot customizations
ax.set_title('Price and Regular Price for Selected Products')

# Place legend below the plot
ax.legend(bbox_to_anchor=(0.5, -0.2), loc='upper center', borderaxespad=0., ncol=1)

ax.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()

# Show the plot
plt.show()